# Representaciones y MLP

**Capítulo 1 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_multilayer-perceptrons/mlp.ipynb` · [Lección original](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Perceptrones multicapa
<a id="sec_mlp"></a>

En [Referencia sec_softmax](https://d2l.ai/chapter_linear-classification/softmax-regression.html#sec-softmax), introdujimos la regresión softmax, implementando el algoritmo desde cero ([Referencia sec_softmax_scratch](https://d2l.ai/chapter_linear-classification/softmax-regression-scratch.html#sec-softmax-scratch)) y usando API de alto nivel ([Referencia sec_softmax_concise](https://d2l.ai/chapter_linear-classification/softmax-regression-concise.html#sec-softmax-concise)), lo que nos permitió entrenar a los clasificadores capaces de reconocer 10 categorías de ropa a partir de imágenes de baja resolución. A lo largo del camino, aprendimos a forcejear datos, coaccionar nuestras salidas hacia una distribución de probabilidad válida, aplicar una función de pérdida apropiada y minimizarla con respecto a los parámetros de nuestro modelo. Ahora que hemos dominado estas mecánicas en el contexto de modelos lineales simples, podemos lanzar nuestra exploración de redes neuronales profundas, la clase comparativamente rica de modelos con la que este libro se refiere principalmente.


In [ ]:
%matplotlib inline
import torch
from laboratorio import d2l

## Capas ocultas
Describimos las transformaciones afín en
[Referencia subsec_linear_model](https://d2l.ai/chapter_linear-regression/linear-regression.html#subsec-linear-model) como
transformaciones lineales con sesgo añadido. Para empezar, recuerde la arquitectura de modelo correspondiente a nuestro ejemplo de regresión softmax, ilustrada en [Referencia fig_softmaxreg](https://d2l.ai/chapter_linear-classification/softmax-regression.html#fig-softmaxreg). Este modelo mapea las entradas directamente a las salidas a través de una única transformación afín, seguida de una operación softmax. Si nuestras etiquetas estaban realmente relacionadas con los datos de entrada por una simple transformación afín, entonces este enfoque sería suficiente. Sin embargo, la linealidad (en transformaciones afín) es una suposición *fuerte*.

### Limitaciones de los modelos lineales
Por ejemplo, la linealidad implica la suposición *más débil* de *monotonicidad*, es decir, que cualquier aumento en nuestra característica debe causar siempre un aumento en la producción de nuestro modelo (si el peso correspondiente es positivo), o siempre causar una disminución en la producción de nuestro modelo (si el peso correspondiente es negativo). A veces eso tiene sentido. Por ejemplo, si estábamos tratando de predecir si un individuo va a pagar un préstamo, podríamos razonablemente suponer que todas las demás cosas son iguales, un solicitante con un ingreso más alto siempre sería más probable que pagar que uno con un ingreso más bajo. Si bien monotónico, esta relación probablemente no está asociada linealmente con la probabilidad de reembolso. Un aumento en los ingresos de \$0 to \$50,000 likely corresponds to a bigger increase in likelihood of repayment than an increase from \$1 million to \$1.05 millones. Una manera de manejar esto podría ser postprocesar nuestro resultado de tal manera que la linealidad se vuelve más plausible, utilizando el mapa logístico (y por lo tanto el logaritm de la probabilidad de resultado).

Tenga en cuenta que podemos encontrar fácilmente ejemplos que violan la monotonicidad. Digamos, por ejemplo, que queremos predecir la salud en función de la temperatura corporal. Para los individuos con una temperatura corporal normal superior a 37°C (98.6°F), las temperaturas más altas indican un mayor riesgo. Sin embargo, si las temperaturas corporales descienden por debajo de 37°C, las temperaturas más bajas indican un mayor riesgo! De nuevo, podríamos resolver el problema con algún preprocesamiento inteligente, como el uso de la distancia de 37°C como una característica.

Pero ¿qué pasa con la clasificación de imágenes de gatos y perros? ¿Debería aumentar la intensidad del píxel en la ubicación (13, 17) siempre aumentar (o siempre disminuir) la probabilidad de que la imagen representa a un perro? La confianza en un modelo lineal corresponde a la suposición implícita de que el único requisito para diferenciar gatos y perros es evaluar el brillo de píxeles individuales. Este enfoque está condenado a fallar en un mundo donde la inversión de una imagen preserva la categoría.

Y sin embargo, a pesar de la aparente absurdidad de la linealidad aquí, en comparación con nuestros ejemplos anteriores, es menos obvio que podríamos abordar el problema con una simple solución de preprocesamiento. Es decir, porque la significación de cualquier píxel depende de formas complejas de su contexto (los valores de los píxeles circundantes). Si bien podría existir una representación de nuestros datos que tendría en cuenta las interacciones relevantes entre nuestras características, encima de lo cual un modelo lineal sería adecuado, simplemente no sabemos cómo calcularlo a mano. Con redes neuronales profundas, utilizamos datos observacionales para aprender conjuntamente tanto una representación a través de capas ocultas y un predictor lineal que actúa sobre esa representación.

Este problema de la no linealidad se ha estudiado durante al menos un siglo [Fisher.1928](https://d2l.ai/chapter_references/zreferences.html). Por ejemplo, los árboles de decisión en su forma más básica utilizan una secuencia de decisiones binarias para decidir sobre la membresía de clase [quinlan2014c4](https://d2l.ai/chapter_references/zreferences.html). Del mismo modo, los métodos de núcleo se han utilizado durante muchas décadas para modelar dependencias no lineales
[Aronszajn.1950](https://d2l.ai/chapter_references/zreferences.html). Esta idea se utiliza en
Modelos no paramétricos de spline [Wahba.1990](https://d2l.ai/chapter_references/zreferences.html) y métodos de núcleo
[Scholkopf.Smola.2002](https://d2l.ai/chapter_references/zreferences.html). El cerebro también resuelve este problema
Después de todo, las neuronas se alimentan de otras neuronas que, a su vez, se alimentan de otras neuronas de nuevo [Cajal.Azoulay.1894](https://d2l.ai/chapter_references/zreferences.html). Consecuentemente tenemos una secuencia de transformaciones relativamente simples.

### Incorporando capas ocultas
Podemos superar las limitaciones de los modelos lineales incorporando una o más capas ocultas. La manera más fácil de hacer esto es apilar muchas capas totalmente conectadas una encima de la otra. Cada capa se alimenta en la capa encima de ella, hasta que generamos salidas. Podemos pensar en las primeras capas $L-1$ como nuestra representación y la capa final como nuestro predictor lineal. Esta arquitectura se llama comúnmente un *perceptor multicapa*, a menudo abreviado como *MLP* ([Referencia fig_mlp](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#fig-mlp)).

![Un MLP con una capa oculta de cinco unidades.](../recursos/originales/mlp.svg)
<a id="fig_mlp"></a>

Este MLP tiene cuatro entradas, tres salidas, y su capa oculta contiene cinco unidades ocultas. Puesto que la capa de entrada no implica ningún cálculo, producir salidas con esta red requiere implementar los cálculos tanto para las capas ocultas como de salida; por lo tanto, el número de capas en este MLP es dos. Tenga en cuenta que ambas capas están totalmente conectadas. Cada entrada influye en cada neurona en la capa oculta, y cada una de estas a su vez influye en cada neurona en la capa de salida.

### De lineal a no lineal
Como antes, denotamos por la matriz $\mathbf{X} \in \mathbb{R}^{n \times d}$ un minibatch de ejemplos $n$ donde cada ejemplo tiene entradas $d$ (características). Para un MLP de una capa oculta cuya capa oculta tiene $h$ unidades ocultas, denotamos por $\mathbf{H} \in \mathbb{R}^{n \times h}$ las salidas de la capa oculta, que son * representaciones ocultas*. Puesto que las capas ocultas y de salida están totalmente conectadas, tenemos pesos $\mathbf{W}^{(1)} \in \mathbb{R}^{d \times h}$ y sesgos $\mathbf{b}^{(1)} \in \mathbb{R}^{1 \times h}$ y pesos $\mathbf{W}^{(2)} \in \mathbb{R}^{h \times q}$ y sesgos $\mathbf{b}^{(2)} \in \mathbb{R}^{1 \times q}$. Esto nos permite calcular las salidas $\mathbf{O} \in \mathbb{R}^{n \times q}$ de la capa oculta MLP como sigue:

$$
\begin{aligned}
    \mathbf{H} & = \mathbf{X} \mathbf{W}^{(1)} + \mathbf{b}^{(1)}, \\
    \mathbf{O} & = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}.
\end{aligned}
$$

Tenga en cuenta que después de añadir la capa oculta, nuestro modelo ahora requiere que rastreemos y actualicemos conjuntos adicionales de parámetros. ¿Qué hemos ganado a cambio? Usted podría sorprenderse al descubrir que---en el modelo definido arriba---* no ganamos nada para nuestros problemas*! La razón es clara. Las unidades ocultas de arriba son dadas por una función afín de las entradas, y las salidas (pre-softmax) son sólo una función afín de las unidades ocultas. Una función afín de una función afín es en sí misma una función afín. Además, nuestro modelo lineal ya era capaz de representar cualquier función afín.

Para ver esto formalmente podemos simplemente colapsar la capa oculta en la definición anterior, produciendo un modelo equivalente de una sola capa con parámetros $\mathbf{W} = \mathbf{W}^{(1)}\mathbf{W}^{(2)}$ y $\mathbf{b} = \mathbf{b}^{(1)} \mathbf{W}^{(2)} + \mathbf{b}^{(2)}$:

$$
\mathbf{O} = (\mathbf{X} \mathbf{W}^{(1)} + \mathbf{b}^{(1)})\mathbf{W}^{(2)} + \mathbf{b}^{(2)} = \mathbf{X} \mathbf{W}^{(1)}\mathbf{W}^{(2)} + \mathbf{b}^{(1)} \mathbf{W}^{(2)} + \mathbf{b}^{(2)} = \mathbf{X} \mathbf{W} + \mathbf{b}.
$$

Con el fin de realizar el potencial de las arquitecturas multicapa, necesitamos un ingrediente clave más: una función de activación no lineal * $\sigma$ que se aplicará a cada unidad oculta después de la transformación afín. Por ejemplo, una opción popular es la función de activación ReLU (unidad lineal rectificada) [Nair.Hinton.2010](https://d2l.ai/chapter_references/zreferences.html) $\sigma(x) = \mathrm{max}(0, x)$ que opera en sus argumentos en sentido de elementos. Las salidas de las funciones de activación $\sigma(\cdot)$ se llaman *activaciones *. En general, con las funciones de activación en su lugar, ya no es posible colapsar nuestro MLP en un modelo lineal:

$$
\begin{aligned}
    \mathbf{H} & = \sigma(\mathbf{X} \mathbf{W}^{(1)} + \mathbf{b}^{(1)}), \\
    \mathbf{O} & = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}.\\
\end{aligned}
$$

Puesto que cada fila en $\mathbf{X}$ corresponde a un ejemplo en el minibatch, con algún abuso de notación, definimos la no linealidad $\sigma$ para aplicar a sus entradas de una manera de fila, es decir, un ejemplo a la vez. Tenga en cuenta que usamos la misma notación para softmax cuando denotamos una operación de fila en [Referencia subsec_softmax_vectorization](https://d2l.ai/chapter_linear-classification/softmax-regression.html#subsec-softmax-vectorization). Muy frecuentemente las funciones de activación que usamos aplican no sólo rowwise sino el elemento. Eso significa que después de calcular la porción lineal de la capa, podemos calcular cada activación sin mirar los valores tomados por las otras unidades ocultas.

Para construir MLPs más generales, podemos seguir apilando tales capas ocultas, por ejemplo, $\mathbf{H}^{(1)} = \sigma_1(\mathbf{X} \mathbf{W}^{(1)} + \mathbf{b}^{(1)})$ y $\mathbf{H}^{(2)} = \sigma_2(\mathbf{H}^{(1)} \mathbf{W}^{(2)} + \mathbf{b}^{(2)})$, una encima de otra, produciendo modelos cada vez más expresivos.

### Aproximadores universales
Sabemos que el cerebro es capaz de un análisis estadístico muy sofisticado. Como tal, vale la pena preguntar, sólo * lo poderoso * una red profunda podría ser. Esta pregunta se ha respondido varias veces, por ejemplo, en [Cybenko.1989](https://d2l.ai/chapter_references/zreferences.html) en el contexto de MLPs, y en [micchelli1984interpolation](https://d2l.ai/chapter_references/zreferences.html) en el contexto de reproducir espacios de Hilbert núcleo de una manera que podría ser visto como función de base radial (RBF) redes con una sola capa oculta. Estos (y resultados relacionados) sugieren que incluso con una sola capa oculta de la red, dados suficientes nodos (posiblemente absurdamente muchos), y el conjunto correcto de pesos, podemos modelar cualquier función. En realidad el aprendizaje de esa función es la parte difícil, aunque. Usted podría pensar que su red neural es un poco como el lenguaje de programación C. El lenguaje, como cualquier otro lenguaje moderno, es capaz de expresar cualquier programa computable. Pero realmente llegar a un programa que cumple con sus especificaciones es la parte difícil.

Por otra parte, el hecho de que una red de una sola capa *puede* aprender cualquier función no significa que usted debe tratar de resolver todos sus problemas con uno. De hecho, en este caso los métodos del núcleo son mucho más eficaces, ya que son capaces de resolver el problema *exactamente* incluso en espacios dimensionales infinitos [Kimeldorf.Wahba.1971,Scholkopf.Herbrich.Smola.2001](https://d2l.ai/chapter_references/zreferences.html). De hecho, podemos aproximar muchas funciones mucho más compactamente mediante el uso de redes más profundas (en lugar de más amplias) [Simonyan.Zisserman.2014](https://d2l.ai/chapter_references/zreferences.html).

## Funciones de activación
<a id="subsec_activation-functions"></a>

Las funciones de activación deciden si una neurona debe activarse o no mediante el cálculo de la suma ponderada y la adición de sesgo a ella. Son operadores diferenciables para transformar las señales de entrada a las salidas, mientras que la mayoría de ellos añaden no linealidad. Debido a que las funciones de activación son fundamentales para el aprendizaje profundo, ** vamos a estudiar brevemente algunos comunes**.

### Función ReLU
La opción más popular, debido a la simplicidad de la implementación y su buen desempeño en una variedad de tareas predictivas, es la *unidad lineal rectificada* (*ReLU*) [Nair.Hinton.2010](https://d2l.ai/chapter_references/zreferences.html). **ReLU proporciona una transformación no lineal muy simple**. Dado un elemento $x$, la función se define como el máximo de ese elemento y $0$:

$$\operatorname{ReLU}(x) = \max(x, 0).$$

Informalmente, la función ReLU sólo retiene elementos positivos y descarta todos los elementos negativos al establecer las activaciones correspondientes a 0. Para obtener alguna intuición, podemos trazar la función. Como puedes ver, la función de activación es lineal por partes.


In [ ]:
x = torch.arange(-8.0, 8.0, 0.1, requires_grad=True)
y = torch.relu(x)
d2l.plot(x.detach(), y.detach(), 'x', 'relu(x)', figsize=(5, 2.5))

Cuando la entrada es negativa, la derivada de la función ReLU es 0, y cuando la entrada es positiva, la derivada de la función ReLU es 1. Tenga en cuenta que la función ReLU no es diferenciable cuando la entrada toma valor exactamente igual a 0. En estos casos, por defecto a la derivada del lado izquierdo y decir que la derivada es 0 cuando la entrada es 0. Podemos salirse con la nuestra porque la entrada puede nunca ser realmente cero (matemáticos dirían que no es diferenciable en un conjunto de medida cero). Hay un viejo adagio que si las condiciones de límite sutil importan, probablemente estamos haciendo (*real*) matemáticas, no ingeniería. Esa sabiduría convencional puede aplicarse aquí, o al menos, el hecho de que no estamos realizando optimización limitada [Mangasarian.1965,Rockafellar.1970](https://d2l.ai/chapter_references/zreferences.html).


In [ ]:
y.backward(torch.ones_like(x), retain_graph=True)
d2l.plot(x.detach(), x.grad, 'x', 'gradiente de ReLU', figsize=(5, 2.5))

La razón para usar ReLU es que sus derivados se comportan particularmente bien: o desaparecen o simplemente dejan pasar el argumento. Esto hace que la optimización se comporte mejor y mitiga el problema bien documentado de los gradientes de desaparición que plagaron versiones anteriores de redes neuronales (más sobre esto más adelante).

Tenga en cuenta que hay muchas variantes a la función ReLU, incluyendo la función *parametrizada ReLU* (*pReLU*) [He.Zhang.Ren.ea.2015](https://d2l.ai/chapter_references/zreferences.html). Esta variación añade un término lineal a ReLU, por lo que todavía se obtiene cierta información, incluso cuando el argumento es negativo:

$$\operatorname{pReLU}(x) = \max(0, x) + \alpha \min(0, x).$$

### Función sigmoide
** La función *sigmoid* transforma aquellas entradas** cuyos valores se encuentran en el dominio $\mathbb{R}$, **a salidas que se encuentran en el intervalo (0, 1).** Por esa razón, el sigmoid a menudo se llama función *squashing*: aplasta cualquier entrada en el rango (-inf, inf) a algún valor en el rango (0, 1):

$$\operatorname{sigmoid}(x) = \frac{1}{1 + \exp(-x)}.$$

En las primeras redes neuronales, los científicos estaban interesados en modelar neuronas biológicas que *fueguen* o *no disparen*. Así, los pioneros de este campo, yendo todo el camino de regreso a McCulloch y Pitts, los inventores de la neurona artificial, enfocados en las unidades de umbral [McCulloch.Pitts.1943](https://d2l.ai/chapter_references/zreferences.html). Una activación de umbral toma valor 0 cuando su entrada está por debajo de algún umbral y valor 1 cuando la entrada excede el umbral.

Cuando la atención se desplazó al aprendizaje basado en gradiente, la función sigmoide fue una opción natural porque es una aproximación suave y diferenciable a una unidad de umbral. Los sigmoids siguen siendo ampliamente utilizados como funciones de activación en las unidades de salida cuando queremos interpretar las salidas como probabilidades de problemas de clasificación binarios: se puede pensar en el sigmoid como un caso especial del softmax. Sin embargo, el sigmoid ha sido reemplazado en gran medida por el ReLU más simple y más fácil de entrenar para la mayoría de uso en capas ocultas. Gran parte de esto tiene que ver con el hecho de que el sigmoid plantea desafíos para la optimización
[LeCun.Bottou.Orr.ea.1998](https://d2l.ai/chapter_references/zreferences.html) porque su gradiente se desvanece tanto para argumentos positivos como negativos de gran magnitud.
Esto puede conducir a mesetas de las que es difícil escapar. Sin embargo, los sigmoids son importantes. En capítulos posteriores (por ejemplo, [Referencia sec_lstm](https://d2l.ai/chapter_recurrent-modern/lstm.html#sec-lstm)) sobre redes neuronales recurrentes, describiremos arquitecturas que aprovechan las unidades sigmoid para controlar el flujo de información a través del tiempo.

A continuación, trazamos la función sigmoide. Tenga en cuenta que cuando la entrada está cerca de 0, la función sigmoide se acerca a una transformación lineal.


In [ ]:
y = torch.sigmoid(x)
d2l.plot(x.detach(), y.detach(), 'x', 'sigmoid(x)', figsize=(5, 2.5))

### Nota docente de Hespérides

Una forma correcta no garantiza un significado correcto: fija qué representa cada eje antes de calcular. En el primer entrenamiento, distingue logits, probabilidades y etiquetas. La ilustración presenta una intuición; el explorador final muestra coordenadas realmente calculadas por una red pequeña.

Vínculo con los apuntes: sesión 1, «Representaciones y MLP».


La derivada de la función sigmoide es dada por la siguiente ecuación:

$$\frac{d}{dx} \operatorname{sigmoid}(x) = \frac{\exp(-x)}{(1 + \exp(-x))^2} = \operatorname{sigmoid}(x)\left(1-\operatorname{sigmoid}(x)\right).$$

La derivada de la función sigmoide se traza a continuación. Tenga en cuenta que cuando la entrada es 0, la derivada de la función sigmoide alcanza un máximo de 0,25. A medida que la entrada diverge de 0 en cualquier dirección, la derivada se aproxima a 0.


In [ ]:
# Limpiar gradientes anteriores
x.grad.data.zero_()
y.backward(torch.ones_like(x),retain_graph=True)
d2l.plot(x.detach(), x.grad, 'x', 'gradiente de sigmoid', figsize=(5, 2.5))

### Función Tanh
<a id="subsec_tanh"></a>

Al igual que la función sigmoide, **la función tanh (tangente hiperbólica) también aplasta sus entradas**, transformándolas en elementos en el intervalo **entre $-1$ y $1$**:

$$\operatorname{tanh}(x) = \frac{1 - \exp(-2x)}{1 + \exp(-2x)}.$$

A continuación trazamos la función tanh. Tenga en cuenta que a medida que la entrada se acerca a 0, la función tanh se aproxima a una transformación lineal. Aunque la forma de la función es similar a la de la función sigmoide, la función tanh muestra simetría de puntos sobre el origen del sistema de coordenadas [Kalman.Kwasny.1992](https://d2l.ai/chapter_references/zreferences.html).


In [ ]:
y = torch.tanh(x)
d2l.plot(x.detach(), y.detach(), 'x', 'tanh(x)', figsize=(5, 2.5))

La derivada de la función tanh es:

$$\frac{d}{dx} \operatorname{tanh}(x) = 1 - \operatorname{tanh}^2(x).$$

A medida que la entrada se acerca a 0, la derivada de la función tanh se aproxima a un máximo de 1. Y como vimos con la función sigmoide, a medida que la entrada se aleja de 0 en cualquier dirección, la derivada de la función tanh se aproxima a 0.


In [ ]:
# Limpiar gradientes anteriores
x.grad.data.zero_()
y.backward(torch.ones_like(x),retain_graph=True)
d2l.plot(x.detach(), x.grad, 'x', 'gradiente de tanh', figsize=(5, 2.5))

## Resumen y debate
Ahora sabemos cómo incorporar no linealidades para construir arquitecturas de red neural de múltiples capas expresivas. Como nota adicional, su conocimiento ya le pone al mando de un conjunto de herramientas similares a un profesional circa 1990. De alguna manera, usted tiene una ventaja sobre cualquier persona que trabaja en ese entonces, porque usted puede aprovechar bibliotecas de aprendizaje profundo de código abierto potentes para construir modelos rápidamente, utilizando sólo unas pocas líneas de código. Anteriormente, la formación de estas redes requería investigadores para codificar capas y derivados explícitamente en C, Fortran, o incluso Lisp (en el caso de LeNet).

Un beneficio secundario es que ReLU es significativamente más susceptible a la optimización que el sigmoid o la función tanh. Se podría argumentar que esta fue una de las innovaciones clave que ayudó al resurgimiento del aprendizaje profundo durante la última década. No obstante, note que la investigación en las funciones de activación no se ha detenido. Por ejemplo, la función de activación GELU (unidad lineal de error gaussiano) $x \Phi(x)$ por [Hendrycks.Gimpel.2016](https://d2l.ai/chapter_references/zreferences.html) ($\Phi(x)$ es la función de distribución acumulativa gaussiana estándar) y la función de activación Swish $\sigma(x) = x \operatorname{sigmoid}(\beta x)$ como se propone en [Ramachandran.Zoph.Le.2017](https://d2l.ai/chapter_references/zreferences.html) puede dar una mejor precisión en muchos casos.

## Ejercicios
1. Mostrar que la adición de capas a una red profunda *lineal*, es decir, una red sin no linealidad $\sigma$ nunca puede aumentar el poder expresivo de la red. Dé un ejemplo donde la reduce activamente.
1. Calcular la derivada de la función de activación pReLU.
1. Calcular la derivada de la función de activación Swish $x \operatorname{sigmoid}(\beta x)$.
1. Demostrar que un MLP usando solo ReLU (o pReLU) construye una función lineal continua a partir de piezas.
1. Sigmoide y tanh son muy similares.
    1. Muestra esa $\operatorname{tanh}(x) + 1 = 2 \operatorname{sigmoid}(2x)$.
    1. Demostrar que las clases de funciones parametrizadas por ambas no linealidades son idénticas. Consejo: las capas afín también tienen términos sesgados.
1. Supongamos que tenemos una no linealidad que se aplica a un minibatch a la vez, como la normalización por lotes (BatchNorm) [Ioffe.Szegedy.2015](https://d2l.ai/chapter_references/zreferences.html). ¿Qué tipo de problemas esperas que cause esto?
1. Proporcionar un ejemplo donde los gradientes desaparecen para la función de activación sigmoide.


[Debate del original](https://discuss.d2l.ai/t/91)
